# Reaktive Maschinenagenten mit Mesa (v3.x)

In diesem Notebook wirst du:
- **Mesa 3.x** installieren,
- einen **reaktiven Agenten** implementieren, der die Temperatur einer Maschine überwacht,
- ein einfaches **Fabrikmodell** mit mehreren Maschinenagenten aufbauen,
- eine **Visualisierung mit Mesa 3.x** im Notebook oder Browser starten und verschiedene Parameter erkunden.


## 1. Installation
Zunächst installieren wir eine aktuelle Mesa-Version **ab 3.0** inklusive Visualisierungsunterstützung.


In [ ]:
#!pip install -U "mesa[viz]>=3.0"

## 2. Mesa-Grundlagen
Mesa-Modelle bestehen aus drei Hauptteilen:
- einer **Model**-Klasse, die den globalen Zustand verwaltet,
- einer oder mehreren **Agent**-Klassen, die das Verhalten einzelner Agenten definieren,
- einer optionalen **Visualisierung**, um das Modell während der Ausführung zu beobachten.

In diesem Beispiel bauen wir einen einfachen *reaktiven* Agenten:
- Er misst seine eigene Temperatur.
- Er wendet eine **Schwellenwert-Regel** an: Liegt die Temperatur über einem Grenzwert, wechselt der Agent in den Zustand `"HOT"`, andernfalls bleibt er abhängig von der Temperatur in `"WARM"`, `"COOL"` oder `"OK"`.


## 3. Implementierung des `MachineAgent`
Der `MachineAgent` repräsentiert eine einzelne Maschine in einer Fabrik. Er besitzt eine Temperatur und eine einfache reaktive Regel.


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Reaktiver Agent, der eine Maschine repräsentiert und deren Temperatur überwacht.

    Regel:
        if temperature > threshold -> state = 'HOT'
        elif temperature > 0.75 * threshold -> state = 'WARM'
        elif temperature < 20 -> state = 'COOL'
        else -> state = 'OK'
    """

    def __init__(self, model, threshold=70):
        super().__init__(model)
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"

    def sense_temperature(self):
        """Einfaches Sensormodell: Temperatur + zufälliges Rauschen."""
        if self.state != "HOT":
            noise = random.uniform(-2, 4)
            self.temperature = self.temperature + noise

    def decide(self):
        """Reaktive Entscheidungsregel, die nur auf der aktuellen Temperatur basiert."""
        if self.temperature > self.threshold:
            self.state = "HOT"
        elif self.temperature > 0.75 * self.threshold:
            self.state = "WARM"
        elif self.temperature < 20:
            self.state = "COOL"
        else:
            self.state = "OK"

    def act(self):
        """Maschine abkühlen (z. B. Simulation einer Drosselung der Maschine)."""
        if self.state == "HOT":
            self.temperature = self.temperature - 10

    def step(self):
        """Agentenschritt = wahrnehmen -> entscheiden -> handeln."""
        self.sense_temperature()
        self.decide()
        self.act()


## 4. Implementierung des `FactoryModel`
Das Modell platziert mehrere Maschinen auf einem Gitter.
Außerdem verwenden wir einen **DataCollector**, um zu verfolgen, wie viele Maschinen sich in den Zuständen `"HOT"` und `"WARM"` befinden.


In [ ]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if a.state == "HOT")


def count_warm_machines(model):
    return sum(1 for a in model.agents if a.state == "WARM")


class FactoryModel(Model):
    """Einfaches Fabrikmodell mit einem Gitter aus Maschinenagenten."""

    def __init__(self, width=10, height=10, density=0.3, threshold=70, seed=None):
        super().__init__(seed=seed)
        self.width = width
        self.height = height
        self.density = density
        self.threshold = threshold

        self.grid = MultiGrid(width, height, torus=False)

        for x in range(self.width):
            for y in range(self.height):
                if self.random.random() < self.density:
                    agent = MachineAgent(self, threshold=self.threshold)
                    self.grid.place_agent(agent, (x, y))

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines": count_hot_machines,
                "WarmMachines": count_warm_machines,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


## 5. Visualisierung mit Mesa 3.x
Mesa 3.x verwendet die neue Visualisierung über `SolaraViz`. Damit funktioniert die Darstellung deutlich besser in modernen Notebook- und Browser-Workflows als der alte Tornado-Server.

Die Schritte sind:
1. Eine **Darstellungsfunktion** (`agent_portrayal`) definieren.
2. Eine Modellinstanz oder Modellklasse mit Parametern an `SolaraViz` übergeben.
3. Die Visualisierung im Notebook anzeigen oder über `solara run` als App starten.

⚠️ **Hinweis:** Falls die Anzeige im Notebook nicht direkt funktioniert, speichere den Code in eine Python-Datei und starte ihn mit `solara run datei.py`.


In [ ]:
from mesa.visualization import SolaraViz, make_space_component, make_plot_component
from mesa.visualization.utils import Slider


def agent_portrayal(agent):
    color = {
        "HOT": "red",
        "WARM": "orange",
        "COOL": "skyblue",
        "OK": "green",
    }.get(agent.state, "gray")

    return {
        "color": color,
        "size": 60,
        "marker": "s",
        "alpha": 0.9,
    }


model_params = {
    "width": 10,
    "height": 10,
    "density": Slider("Dichte", 0.3, 0.1, 1.0, 0.1),
    "threshold": Slider("Temperatur-Schwelle", 70, 30, 100, 5),
}

space_component = make_space_component(agent_portrayal)
plot_component = make_plot_component(["HotMachines", "WarmMachines"])

page = SolaraViz(
    FactoryModel,
    components=[space_component, plot_component],
    model_params=model_params,
    name="Reaktive Maschinenagenten",
)

page

## 6. Simulation ohne Visualisierung testen
Falls du nur die Modelllogik testen möchtest, kannst du das Modell auch direkt einige Schritte laufen lassen.


In [ ]:
model = FactoryModel(width=10, height=10, density=0.3, threshold=70, seed=42)
for _ in range(20):
    model.step()

model.datacollector.get_model_vars_dataframe().tail()